In [1]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import accuracy_score
import joblib

In [2]:
df_non_scaled=pd.read_csv("cleaned.csv")
df_non_scaled.head()

,a1_std,a2_std,a3_std,a4_std,a1_min,a2_min,a3_min,a4_min,a1_max,a2_max,...,a1_kurtosis,a2_kurtosis,a3_kurtosis,a4_kurtosis,a1_skew,a2_skew,a3_skew,a4_skew,load,failure
0,2.773743,3.625441,3.053217,2.479670,-6.75195,-8.78230,-6.87031,-4.42552,6.94606,7.58304,...,-0.323687,0.004087,0.099096,1.108623,-0.257241,-0.162347,0.124041,0.622273,0.0,1.0
1,3.009118,3.566201,2.854556,2.597574,-6.79768,-8.27673,-6.90384,-7.08858,6.18348,9.62947,...,-0.685575,-0.165116,0.275550,0.387465,-0.276394,0.095039,0.346980,-0.098255,0.0,1.0
2,2.258893,3.189251,1.971267,1.726624,-6.03566,-8.72943,-4.80122,-5.43327,5.48630,7.11868,...,0.022196,0.066009,-0.014646,0.844893,-0.172785,-0.203286,0.119108,-0.233903,0.0,1.0
3,2.486789,2.758437,2.658690,2.068924,-5.98302,-6.96731,-8.49161,-3.86559,4.91734,9.08417,...,-0.335753,0.799129,0.765071,2.535101,-0.197734,-0.073012,-0.001285,0.727516,0.0,1.0
4,3.245092,3.051541,2.536531,2.085758,-9.57229,-11.21170,-7.04513,-4.19897,9.27243,8.73261,...,1.059798,1.290685,0.231211,1.418064,-0.220589,-0.212601,-0.269316,0.947665,0.0,1.0


In [3]:
X_non_scaled=df_non_scaled.drop("failure",axis=1)
Y=df_non_scaled["failure"]

X_non_scaled.head()

,a1_std,a2_std,a3_std,a4_std,a1_min,a2_min,a3_min,a4_min,a1_max,a2_max,...,a4_range,a1_kurtosis,a2_kurtosis,a3_kurtosis,a4_kurtosis,a1_skew,a2_skew,a3_skew,a4_skew,load
0,2.773743,3.625441,3.053217,2.479670,-6.75195,-8.78230,-6.87031,-4.42552,6.94606,7.58304,...,14.35893,-0.323687,0.004087,0.099096,1.108623,-0.257241,-0.162347,0.124041,0.622273,0.0
1,3.009118,3.566201,2.854556,2.597574,-6.79768,-8.27673,-6.90384,-7.08858,6.18348,9.62947,...,13.84640,-0.685575,-0.165116,0.275550,0.387465,-0.276394,0.095039,0.346980,-0.098255,0.0
2,2.258893,3.189251,1.971267,1.726624,-6.03566,-8.72943,-4.80122,-5.43327,5.48630,7.11868,...,9.78933,0.022196,0.066009,-0.014646,0.844893,-0.172785,-0.203286,0.119108,-0.233903,0.0
3,2.486789,2.758437,2.658690,2.068924,-5.98302,-6.96731,-8.49161,-3.86559,4.91734,9.08417,...,13.32480,-0.335753,0.799129,0.765071,2.535101,-0.197734,-0.073012,-0.001285,0.727516,0.0
4,3.245092,3.051541,2.536531,2.085758,-9.57229,-11.21170,-7.04513,-4.19897,9.27243,8.73261,...,11.04642,1.059798,1.290685,0.231211,1.418064,-0.220589,-0.212601,-0.269316,0.947665,0.0


In [4]:
Y.head()

0    1.0
1    1.0
2    1.0
3    1.0
4    1.0
Name: failure, dtype: float64

In [5]:
X_non_scaled.shape

(20213, 29)

In [6]:
Y.shape

(20213,)

In [7]:
scaler=StandardScaler()
temp=scaler.fit_transform(X_non_scaled)
X=pd.DataFrame(temp,columns=X_non_scaled.columns)
X.head()

,a1_std,a2_std,a3_std,a4_std,a1_min,a2_min,a3_min,a4_min,a1_max,a2_max,...,a4_range,a1_kurtosis,a2_kurtosis,a3_kurtosis,a4_kurtosis,a1_skew,a2_skew,a3_skew,a4_skew,load
0,-1.352116,-0.552798,-0.779382,-1.465253,1.256305,0.836606,1.160637,1.923937,-1.358475,-0.939330,...,-1.326594,-0.789526,-0.537573,-0.643797,0.363085,-1.156686,0.053279,0.587176,1.823279,-1.592817
1,-1.245847,-0.603850,-0.974248,-1.369137,1.249712,0.960107,1.151005,1.231305,-1.476224,-0.347935,...,-1.400927,-1.132050,-0.703734,-0.482793,-0.322120,-1.225050,0.808009,1.248668,-0.091159,-1.592817
2,-1.584567,-0.928698,-1.840662,-2.079135,1.359571,0.849521,1.755035,1.661833,-1.583875,-1.073525,...,-1.989335,-0.462150,-0.476764,-0.747580,0.112503,-0.855228,-0.066766,0.572540,-0.451573,-1.592817
3,-1.481673,-1.299967,-1.166372,-1.800092,1.367160,1.279974,0.694878,2.069569,-1.671728,-0.505521,...,-1.476576,-0.800945,0.243175,-0.036135,1.718444,-0.944279,0.315235,0.215317,2.102906,-1.592817
4,-1.139306,-1.047375,-1.286197,-1.786370,0.849702,0.243149,1.110416,1.982860,-0.999262,-0.607117,...,-1.807016,0.519932,0.725893,-0.523250,0.657098,-1.025859,-0.094081,-0.579969,2.687842,-1.592817


In [8]:
X.columns

Index(['a1_std', 'a2_std', 'a3_std', 'a4_std', 'a1_min', 'a2_min', 'a3_min',
       'a4_min', 'a1_max', 'a2_max', 'a3_max', 'a4_max', 'a1_var', 'a2_var',
       'a3_var', 'a4_var', 'a1_range', 'a2_range', 'a3_range', 'a4_range',
       'a1_kurtosis', 'a2_kurtosis', 'a3_kurtosis', 'a4_kurtosis', 'a1_skew',
       'a2_skew', 'a3_skew', 'a4_skew', 'load'],
      dtype='object')

In [9]:
pipeline=Pipeline([("classifier",RandomForestClassifier())])

grid=[{
    "classifier":[RandomForestClassifier()],
    "classifier__n_estimators":[50,100,150,200,250],
    "classifier__max_depth":[None,5,10]
},{
    "classifier":[XGBClassifier()],
    "classifier__n_estimators":[50,100,150,200,250,500],
    "classifier__max_depth":[None,5,10],
    "classifier__learning_rate":[0.05,0.1,0.3,0.5]
}]

In [10]:
gs=GridSearchCV(pipeline,grid,cv=5,scoring="f1",n_jobs=-1)
gs.fit(X,Y)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...lassifier())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","[{'classifier': [RandomForestClassifier()], 'classifier__max_depth': [None, 5, ...], 'classifier__n_estimators': [50, 100, ...]}, {'classifier': [XGBClassifier...ree=None, ...)], 'classifier__learning_rate': [0.05, 0.1, ...], 'classifier__max_depth': [None, 5, ...], 'classifier__n_estimators': [50, 100, ...]}]"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv``

In [13]:
print(f"Best paramaeters are {gs.best_params_} with a score of {gs.best_score_}")

Best paramaeters are {'classifier': XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...), 'classifier__learning_rate': 0.3, 'classifier__max_depth': 10, 'classifier__n_estimators': 500} with a score of 0.9449216548829682


In [14]:
results=pd.DataFrame(gs.cv_results_)
results.head()

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_classifier,param_classifier__max_depth,param_classifier__n_estimators,param_classifier__learning_rate,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,3.755721,0.327034,0.038431,0.010386,RandomForestClassifier(),None,50,NaN,"{'classifier': RandomForestClassifier(), 'clas...",0.939213,0.945044,0.935939,0.927720,0.935118,0.936607,0.005650,76
1,7.081322,0.318662,0.057039,0.006114,RandomForestClassifier(),None,100,NaN,"{'classifier': RandomForestClassifier(), 'clas...",0.941092,0.947033,0.935476,0.928065,0.940665,0.938466,0.006360,71
2,10.785249,0.415414,0.082640,0.009479,RandomForestClassifier(),None,150,NaN,"{'classifier': RandomForestClassifier(), 'clas...",0.941429,0.947188,0.938071,0.928499,0.943396,0.939717,0.006338,66
3,14.014195,0.553011,0.108952,0.010334,RandomForestClassifier(),None,200,NaN,"{'classifier': RandomForestClassifier(), 'clas...",0.941317,0.946516,0.938765,0.928300,0.942229,0.939425,0.006098,68
4,17.798082,0.918874,0.142914,0.008276,RandomForestClassifier(),None,250,NaN,"{'classifier': RandomForestClassifier(), 'clas...",0.941148,0.948479,0.936959,0.929349,0.938974,0.938982,0.006193,69


In [15]:
best_idx = gs.best_index_

print("Fold 1:", results.loc[best_idx, 'split0_test_score'])
print("Fold 2:", results.loc[best_idx, 'split1_test_score'])
print("Fold 3:", results.loc[best_idx, 'split2_test_score'])
print("Fold 4:", results.loc[best_idx, 'split3_test_score'])
print("Fold 5:", results.loc[best_idx, 'split4_test_score'])

print("Mean:", results.loc[best_idx, 'mean_test_score'])

Fold 1: 0.9438148058837714
Fold 2: 0.9461462450592886
Fold 3: 0.9442211055276382
Fold 4: 0.9399310684391925
Fold 5: 0.9504950495049505
Mean: 0.9449216548829682


In [16]:
best_model=XGBClassifier(n_estimators=500,learning_rate=0.3,max_depth=10)
best_model.fit(X,Y)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

In [23]:
joblib.dump(best_model,"model.pkl")

['model.pkl']

In [24]:
joblib.dump(X.columns,"columns.pkl")

['columns.pkl']

In [25]:
joblib.dump(scaler,"scaler.pkl")

['scaler.pkl']